In [21]:
import numpy as np
import math
from scipy.special import factorial
from numpy.random import shuffle
import matplotlib.pyplot as plt
from os import listdir

#### Objective function

$$
O_K=\sum_{n=n_0}^{n_0+K^{-1}} \sum_{i \in C_n} \frac{1}{N_t} \sum_t\left[V_i(t)-\bar{V}_n(t)\right]^2 
$$


In [14]:
config = {
    'Dataset_path'          : "Datasets/NIST (250)",
    'Dataset_signal_size'   : 250,
    'Average_photon_number' : [2,3,4,5],
    'Objective_sigma'       : 1.46,
    'Optimization_iter'     : 10
}

In [15]:
class PIKA():

    def __init__(self, config):

        # Dataset 
        self.folder = config['Dataset_path']
        self.size = config['Dataset_signal_size']
        
        # Initial assumptions
        self.N_mean_list = config['Average_photon_number']
        self.objective_sigma = config['Objective_sigma']

        # Loop iteration 
        self.optimization_iter = config['Optimization_iter']
        self.nb_cluster = 0

        # Signal parameters
        self.nb_samples = 0
        self.N_mean = 0

        # Objective history
        self.KMean_Objective_hist = []
        
        

    def Get_traces(self):

        V = np.concatenate([np.fromfile(f"{self.folder}/{file_name}", dtype=np.float16).reshape((-1,self.size)) for file_name in listdir(self.folder)])
        self.nb_samples = len(V)
        index = np.arange(self.nb_samples)

        return V, index




    def Dot_product(self, V, index):
        V_mean = np.mean(V, axis = 0)
        n_eff = self.N_mean * np.sum(V_mean * V, axis=1) / np.sum(V_mean * V_mean)

        _, index = zip(*sorted(zip(n_eff, index)))

        return index

    def Poisson(self, index):
        # Compute 95% confidence interval
        sigma = np.sqrt(self.N_mean)
        upper_limit = math.ceil(self.N_mean + 1.3 * sigma)
        lower_limit = math.floor(self.N_mean - 1.3 * sigma)
        if lower_limit < 0: lower_limit = 0

        N = np.arange(lower_limit, upper_limit + 1)
        prob = np.absolute( np.exp(-self.N_mean) * self.N_mean**N / factorial(N) )
        int_prob = np.rint(prob / np.sum(prob) * self.nb_samples).astype(int)

        N = N[int_prob > 0]
        int_prob = int_prob[int_prob > 0]
        

        int_prob[np.argmax(int_prob)] += self.nb_samples - np.sum(int_prob)

        split = [int(np.sum(int_prob[:i+1])) for i, _ in enumerate(int_prob)]

        return N , np.split(index, split)[:-1]


    def Initial_cluster(self, V, index):
        index = self.Dot_product(V, index)
        N, clusters = self.Poisson(index)
        self.nb_cluster = len(clusters)
        
        return N, clusters

        

    def KMeans_objective(self, V, clusters):
        O_K = 0
        for n, _ in enumerate(clusters):
            V_n = V[clusters[n]]
            V_n_mean = np.mean(V_n, axis = 0)
            cluster_O_K = 1/self.size * np.sum((V_n - V_n_mean)**2))
            O_K += cluster_O_K
            self.KMean_Objective_hist.append(cluster_O_K)

        return O_K


    def Poisson_likelihood(self, N, clusters):
        L_P = 1
        for index, N in enumerate(N):
            m_p = len(clusters[index])
            L_P *= (np.exp(-self.N_mean) * self.N_mean**N / factorial(N))**m_p

        return L_P

    def Combinatorial_likelihood(self, N, clusters):
        L_C_inv = 1
        for index, N in enumerate(N):
            m_p = len(clusters[index])
            L_C_inv *= factorial(m_p)

        return self.nb_samples / L_C_inv

    def Poisson_Combinatorial_objective(self, N, clusters):
        
        return -np.log(self.Poisson_likelihood(N, clusters) * self.Combinatorial_likelihood(N, clusters))

    
    def KMeans_Poisson_objective(self, V, N, clusters):
        O_K = self.KMeans_objective(V, clusters)
        O_PC = self.Poisson_Combinatorial_objective(N, clusters)

        return 1/(2*self.objective_sigma**2) * O_K + O_PC


    def Randomize_clusters(self, clusters):

        for index, cluster in enumerate(clusters):
            shuffle(cluster)
            clusters[index] = cluster

        return clusters

    
    def Update_Objective



    def run_PIKA(self):

        V, index = self.get_traces()

        for self.N_mean in self.N_mean_list:

            N, clusters = self.initial_cluster(V, index)
            O_KPC = self.KMeans_Poisson_objective(V, N, clusters)

            for opt_idx in range(self.optimization_iter):

                clusters = self.Randomize_clusters(clusters)

                for cluster_idx in range(self.nb_cluster):
                    
                    
                    
            
            
            

     
        

SyntaxError: invalid syntax (1590378285.py, line 123)

In [ ]:
alg = PIKA(config)
O_k = alg.run_PIKA()

O_k

2


C:\Users\dalbe\AppData\Local\Temp\ipykernel_36060\1710084396.py:92: RuntimeWarning: divide by zero encountered in log
  return -np.log(self.Poisson_likelihood(N, clusters) * self.Combinatorial_likelihood(N, clusters))


3
4
5


inf

In [24]:
x = np.array([[1,2,3,4,5],[6,7,8,9]])
shuffle(x)
print(x)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.